**常用函数**
功能	    函数/方法	              说明
读取视频	    cv2.VideoCapture()	  读取视频文件或摄像头。
逐帧读取视频	cap.read()	          逐帧读取视频。
获取视频属性	cap.get(propId)	      获取视频的属性（如宽度、高度、帧率等）。
保存视频	    cv2.VideoWriter()	  创建视频写入对象并保存视频。
视频帧处理	图像处理函数（如 cv2.cvtColor()）	对视频帧进行图像处理。
目标跟踪	    cv2.TrackerKCF_create()	使用目标跟踪算法跟踪视频中的物体。
运动检测	    cv2.createBackgroundSubtractorMOG2()	使用背景减除算法检测视频中的运动物体。

打开原视频、灰色视频、摄像头、物品检测（打框标注）、视频追踪（框框跟踪）、视频背景减除（只有黑白）

打开视频

In [15]:
import cv2

# 创建 VideoCapture 对象，读取摄像头视频
# （0）为本地摄像机 /（“....MP4”）导入视频
cap = cv2.VideoCapture(0)

# 检查摄像头是否成功打开
if not cap.isOpened():
    print("Error: Could not open camera.")
    exit()

# 读取视频帧
while True:
    ret, frame = cap.read()
    
    # 如果读取到最后一帧，退出循环
    if not ret:
        break
    
    # 显示当前帧
    cv2.imshow('Camera', frame)
    
    # 按下 'q' 键退出（会爆红 但是不影响弹出相机框框的程序）
    # 不算代码bug是被迫终止程序
    # 运行25毫秒后可以按‘q’退出
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

# 释放资源
cap.release()
cv2.destroyAllWindows()

打开摄像头

In [3]:
import cv2
from matplotlib import pyplot as plt

# 创建 VideoCapture 对象，读取视频文件
cap = cv2.VideoCapture('video.mp4')

# 检查视频是否成功打开
if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

# 读取视频帧
while True:
    ret, frame = cap.read()
    
    # 如果读取到最后一帧，退出循环
    if not ret:
        break
    
    # 显示当前帧
    cv2.imshow('Video', frame)
    
    # 按下 'q' 键退出
    # 运行25毫秒后可以按‘q’退出
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

# 释放资源
cap.release()
cv2.destroyAllWindows()

把视频变成灰色

In [1]:
import cv2

cap = cv2.VideoCapture('video.mp4')

while True:
    ret, frame = cap.read()
    
    if not ret:
        break
    
    # 将帧转换为灰度图像
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 显示灰度帧
    cv2.imshow('Gray Video', gray_frame)
    
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

把视频变成灰色，并且把灰色视频保存下来

In [3]:
import cv2

cap = cv2.VideoCapture('video.mp4')

# 获取视频的帧率和尺寸
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 创建 VideoWriter 对象，保存处理后的视频
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    
    if not ret:
        break
    
    # 将帧转换为灰度图像
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 将灰度帧写入输出视频
    out.write(cv2.cvtColor(gray_frame, cv2.COLOR_GRAY2BGR))
    
    # 显示灰度帧
    cv2.imshow('Gray Video', gray_frame)
    
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

视频中的物体检测(检测的有点忒水了)

In [41]:
import cv2

# 加载 Haar 特征分类器
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture('baby.mp4')

while True:
    ret, frame = cap.read()
    
    if not ret:
        break
    
    # 将帧转换为灰度图像
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 检测人脸（不一定是人脸 可能视频识别度问题）
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    
    # 在帧上绘制矩形框标记人脸
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
    
    # 显示带有人脸标记的帧
    cv2.imshow('Face Detection', frame)
    
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

视频中的运动检测（一些看不懂）

In [13]:
import cv2
import numpy as np

# 读取视频
cap = cv2.VideoCapture('baby.mp4')

# 读取第一帧
ret, frame = cap.read()

# 设置初始窗口 (x, y, width, height) 
# 根据框住的内容实时追踪
x, y, w, h = 300, 200, 100, 50    
track_window = (x, y, w, h)

# 设置 ROI (Region of Interest)特别关注区域  框框
roi = frame[y:y+h, x:x+w]

# 转换为 HSV 颜色空间
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# 创建掩膜并计算直方图
mask = cv2.inRange(hsv_roi, np.array((0., 60., 32.)), np.array((180., 255., 255.)))
roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# 设置终止条件
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 转换为 HSV 颜色空间
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 计算反向投影
    dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

    # 应用 MeanShift 算法
    ret, track_window = cv2.meanShift(dst, track_window, term_crit)

    # 绘制跟踪结果
    x, y, w, h = track_window
    img2 = cv2.rectangle(frame, (x, y), (x+w, y+h), 255, 2)
    cv2.imshow('MeanShift Tracking', img2)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

打开摄像头（简易版）

In [14]:
import cv2

# 打开默认摄像头
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow('Frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

会录制摄像头视频 不过保存后不完整？？

In [19]:
import cv2

cap = cv2.VideoCapture(0)
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('sp.avi', fourcc, 20.0, (640, 480))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)
    cv2.imshow('Frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

视频框框追踪（和上面差不多）

In [38]:
import cv2
import numpy as np

# 读取视频
cap = cv2.VideoCapture('baby.mp4')

# 读取第一帧
ret, frame = cap.read()

# 设置初始窗口 (x, y, width, height)
x, y, w, h = 300, 200, 100, 50
track_window = (x, y, w, h)

# 设置 ROI (Region of Interest)
roi = frame[y:y+h, x:x+w]

# 转换为 HSV 颜色空间
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# 创建掩膜并计算直方图
mask = cv2.inRange(hsv_roi, np.array((0., 60., 32.)), np.array((180., 255., 255.)))
roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# 设置终止条件
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 转换为 HSV 颜色空间
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 计算反向投影
    dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

    # 应用 MeanShift 算法
    ret, track_window = cv2.meanShift(dst, track_window, term_crit)

    # 绘制跟踪结果
    x, y, w, h = track_window
    img2 = cv2.rectangle(frame, (x, y), (x+w, y+h), 255, 2)
    cv2.imshow('MeanShift Tracking', img2)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

视频背景减除  成只有黑白的视频了

In [40]:
import cv2

# 读取视频
cap = cv2.VideoCapture("baby.mp4")

# 创建 MOG2 背景减除器
fgbg = cv2.createBackgroundSubtractorMOG2()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 应用背景减除器
    fgmask = fgbg.apply(frame)

    # 显示结果
    cv2.imshow("MOG2 Background Subtraction", fgmask)

    # 按下 'q' 键退出
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# 释放资源
cap.release()
cv2.destroyAllWindows()